In [14]:
import os
import numpy as np
from pathlib import Path
from jiwer import process_words
from rapidfuzz.distance import Levenshtein

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

tg_path = Path("/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/align_w2v")
ref_tg_path = Path("/vol/corpora/Rhapsodie/TextGrids-fev2013")

full_paths = list(tg_path.rglob("*.TextGrid"))
ref_files = list(ref_tg_path.glob("*"))
# ------------------------------------------------------------------
# Alignment helper
# ------------------------------------------------------------------
from praatio import textgrid


def extract_phones_from_textgrid(tg_path, remove_silence=True,t=""):
    """
    Extract phoneme sequence and timestamps from MFA TextGrid.

    Returns:
        phones: list of phoneme labels
        intervals: list of (start, end, phone)
    """
    
    tg = textgrid.openTextgrid(tg_path, includeEmptyIntervals=True)
    
    # List available tiers
   # print("Available tiers:", tg.tierNames)
    
    # Usually MFA phoneme tier is named "phones"
    phone_tier = tg.getTier(t)
    
    phones = []
    intervals = []
    
    for start, end, label in phone_tier.entries:
        
        label = label.strip()
        
        # Skip empty intervals
        if label == "":
            continue
        
        # Optionally remove silence
        if remove_silence and label in ["sil", "sp", "spn"]:
            continue
        
        phones.append(label)
        intervals.append((start, end, label))
    
    return phones, intervals
def extract_phones_from_textgrid_typaloc(textgrid_path, remove_silence=True):

    tg = textgrid.openTextgrid(
        textgrid_path,
        includeEmptyIntervals=False,
        duplicateNamesMode="rename"
    )

    # 🔎 Find tier containing "corr"
    tier_name = None
    for t in tg.tierNames:
        if "corr" in t.lower():
            tier_name = t
            break

    # fallback to first tier if none found
    if tier_name is None:
        tier_name = tg.tierNames[0]

    phone_tier = tg.getTier(tier_name)
    
    phones = []
    intervals = []
    
    for start, end, label in phone_tier.entries:
        
        label = label.strip()
        
        # Skip empty intervals
        if label == "":
            continue
        
        # Optionally remove silence
        if remove_silence and label in ["sil", "sp", "spn"]:
            continue
        
        phones.append(label)
        intervals.append((start, end, label))
    
    return phones, intervals
    
def align_sequences(ref, hyp):
    alignment = []
    ops = Levenshtein.editops(ref, hyp)

    ref_idx = hyp_idx = 0
    op_idx = 0

    while ref_idx < len(ref) or hyp_idx < len(hyp):
        if op_idx < len(ops) and \
           ops[op_idx].src_pos == ref_idx and \
           ops[op_idx].dest_pos == hyp_idx:

            op = ops[op_idx]

            if op.tag == "replace":
                alignment.append((ref_idx, hyp_idx))
                ref_idx += 1
                hyp_idx += 1

            elif op.tag == "delete":
                alignment.append((ref_idx, None))
                ref_idx += 1

            elif op.tag == "insert":
                alignment.append((None, hyp_idx))
                hyp_idx += 1

            op_idx += 1
        else:
            alignment.append((ref_idx, hyp_idx))
            ref_idx += 1
            hyp_idx += 1

    return alignment


# ------------------------------------------------------------------
# Boundary error computation
# ------------------------------------------------------------------

def boundary_errors(ref_intervals, hyp_intervals, alignment,phones_ref,phones_hyp):
    start_errors = []
    end_errors = []
    duration_errors = []

    for ref_idx, hyp_idx in alignment:

        if ref_idx is None or hyp_idx is None:
            continue

        if phones_ref[ref_idx] == phones_hyp[hyp_idx]:

            r_start = ref_intervals[ref_idx]["start"]
            r_end   = ref_intervals[ref_idx]["end"]

            h_start = hyp_intervals[hyp_idx]["start"]
            h_end   = hyp_intervals[hyp_idx]["end"]

            start_errors.append(abs(r_start - h_start))
            end_errors.append(abs(r_end - h_end))
            duration_errors.append(
                abs((r_end - r_start) - (h_end - h_start))
            )

    return start_errors, end_errors, duration_errors


# ------------------------------------------------------------------
# Build reference dictionary (stem → file path)
# ------------------------------------------------------------------

#ref_dict={str(f.stem)[:-4].split("-")[1]+"-"+str(f.stem)[:-4].split("-")[0]:str(f) for f in ref_files}
#ref_dict = {str(f.stem)[:-4]:str(f) for f in ref_files}
#ref_dict = {str(f.stem):str(f) for f in ref_files}
import unicodedata
foreign_phonemes = {
    'β','θ','ɹ','ɾ','ɣ','ʌ','ʊ','ɪ','ɨ','ɨ̃','ɜ','ɒ','õ','ũ'
}
VOWELS = set("aeiouyɛøœɔɑɨɪʊʌɒɜəɛ")
REF_MAPPING = {
    # Consonants
    "Z": "ʒ",
    "A":"a",
    "S": "ʃ",
    "R": "ʁ",
    "r": "ʁ",
    "N": "ŋ",
    "J": "ɲ",
    "H": "ɥ",
    "g": "ɡ",
    "Z=": "ʒ",

    # Vowels
    "E": "ɛ",
    "O": "ɔ",
    "2": "ø",
    "9": "œ",
    "@": "ə",

    # Nasals (SAMPA)
    "a~": "ɑ̃",
    "o~": "ɔ̃",
    "e~": "ɛ̃",
    "9~": "ɛ̃",
    "E": "ɛ",
    "m=": "m",
    "n=": "n",
    "9~": "ɛ̃",
}
NASAL_CANONICAL = {

    "ã": "ɑ̃",
    "ẽ": "ɛ̃",
    "ĩ": "ɛ̃",
    "ỹ": "ɛ̃",
    "œ̃": "ɛ̃",
    "ə̃": "ɛ̃",
}
HYP_PROJECTION = {

    # Multilingual vowel variants
    "ɪ": "i",
    "ʊ": "u",
    "ɨ": "i",
    "ɜ": "ə",
    "ʌ": "ɔ",
    "ɒ": "ɔ",

    # Rhotic variants
    "ɣ": "ʁ",
    "ɹ": "ʁ",
    "ɾ": "ʁ",

    # Lateral variant
    "ʎ": "l",

    # Foreign consonants
    "β": "b",
    "θ": "t",
    "c": "k",
    "ɑ": "a",
    "mʲ":"m",
    "ɟ": "ɲ",
}
asr_to_ipa = {
    "aa": "a",        # a, ɑ
    "bb": "b",
    "kk": "k",        # c, k
    "dd": "d",
    "jj": "dʒ",       # also ʒ (see note below)
    "ei": "e",
    "ff": "f",
    "ii": "i",
    "yy": "j",
    "ll": "l",        # also ʎ
    "mm": "m",        # also mʲ
    "nn": "n",        # also ŋ
    "au": "o",
    "pp": "p",
    "ss": "s",        # also ts
    "tt": "t",
    "ch": "ʃ",        # also tʃ
    "ou": "u",
    "vv": "v",
    "ww": "w",
    "uu": "y",
    "zz": "z",
    "eu": "ø",
    "oe": "œ",
    "an": "ɑ̃",
    "oo": "ɔ",
    "on": "ɔ̃",
    "ee": "ə",
    "ai": "ɛ",
    "in": "ɛ̃",
    "un": "ɛ̃",       # second nasal mapping
    "gn": "ɲ",        # also ɟ depending on system
    "gg": "ɡ",
    "uy": "ɥ",
    "rr": "ʁ",
    "r": "ʁ",
    "SIL": "_"
}
def normalize_phoneme_typaloc(ph):
    if ph is None:
        return None
    # enlever contenu [[...]]
    ph = unicodedata.normalize("NFC", ph)
    ph = re.sub(r"\[\[.*?\]\]", "", ph)

    # enlever crochets restants mal formés
    ph = re.sub(r"\[\[|\]\]", "", ph)

    # enlever contenu entre parenthèses
    ph = re.sub(r"\(.*?\)", "", ph)

    # enlever NONCORR
    ph = re.sub(r"\bNONCORR\b", "", ph)

    # nettoyer espaces
    ph = re.sub(r"\s+", " ", ph).strip()
    
    
    # enlever espaces multiples
    ph = re.sub(r"\s+", " ", ph).strip()
    ph =re.sub(r"\n.*", "", ph, flags=re.DOTALL)
    ph = ph.replace("yu","uy")
    ph = ph.replace("nn+yy","nn")
    ph = ph.replace("ei\t\t","ei")
    if "NB sur tDeb" in ph:
        ph="ei"
    ph = ph.replace("#a","a")
    ph = ph.replace("kk+","k")
    ph = re.sub(r"\*.*?\*", "", ph)
    ph = re.sub(r"\[\s*pause\s*\]", "", ph)
    ph = re.sub(r"\b\w*pause\w*\b", "", ph)
    if ph in ["_", "sil", "spn", "%", "?", "??","0","#", "=","euh","#erreur#"]:
        return None
    # Remove standalone combining marks
    if ph and all(unicodedata.combining(c) for c in ph):
        return None
    # --- Reference mapping ---
    if ph in asr_to_ipa.keys():
        ph = asr_to_ipa[ph]
    ph = ph.replace("dʒ","ʒ")
    return ph
        
def normalize_phoneme(ph, is_hyp=False):

    if ph is None:
        return None
    ph = ph.replace(" ", "")
    ph = re.sub(r"@t", "ə", ph)
    ph = re.sub(r"\?", "", ph)
    ph = re.sub(r"\@", "ə", ph)
    ph = ph.replace("<p:>", "")
    # Unicode normalization
    ph = unicodedata.normalize("NFC", ph)
    # --- Remove silence / junk tokens ---
    if ph in ["_", "sil", "spn", "%", "?", "0", "=", "fe~", "Ra~", "sjo~", "~e"]:
        return None
    

    # --- Remove standalone combining marks ---
    if all(unicodedata.combining(c) for c in ph):
        return None
    
    # --- Reference mapping ---
    if not is_hyp and ph in REF_MAPPING:
        ph = REF_MAPPING[ph]

    # --- Hyp multilingual projection ---
    if is_hyp and ph in HYP_PROJECTION:
        ph = HYP_PROJECTION[ph]

    # --- Collapse multiple nasal marks ---
    ph = re.sub(r"\u0303+", "\u0303", ph)

    # --- Prevent nasalized consonants ---
    if len(ph) > 1:
        base = ph[0]
        if base not in VOWELS:
            ph = base  # remove nasalization from consonants

    # --- Canonical nasal mapping ---
    if ph in NASAL_CANONICAL:
        ph = NASAL_CANONICAL[ph]
    if ph in foreign_phonemes:
        return None
    

    # --- Remove suprasegmentals ---
    ph = ph.replace("ː", "")

    if ph.strip() == "":
        return None

    return ph

def clean_alignment_dict(alignment_list,flag="",is_hyp=False):
    """
    Normalize phonemes and remove empty or deleted ones.
    Keeps timestamps aligned.
    """
    
    cleaned = []
    phonemes =[]
    for item in alignment_list:
        phoneme = item["phoneme"]
        
        # Normalize
        if flag=="typaloc":
            phoneme_norm = normalize_phoneme_typaloc(phoneme)
        else:
            phoneme_norm = normalize_phoneme(phoneme, is_hyp=is_hyp)
        
        # Remove empty phonemes after normalization
        if phoneme_norm == "" or phoneme_norm is None:
            continue
        phonemes.append(phoneme_norm)
        cleaned.append({
            "phoneme": phoneme_norm,
            "start": item["start"],
            "end": item["end"]
        })
    
    return cleaned,phonemes


In [3]:
#typaloc
"""ref_dict = {
    str(f.stem)[:-11]: str(f)
    for f in ref_files
    if str(f.stem).endswith("pr_analyse")
}"""
#rhapsodie
ref_dict={str(f.stem)[:-4].split("-")[1]+"-"+str(f.stem)[:-4].split("-")[0]:str(f) for f in ref_files}

#ref_dict = {str(f.stem)[:-4]:str(f) for f in ref_files}
#ref_dict = {str(f.stem):str(f) for f in ref_files}
ref_dict

{'D0020-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D0020-Pro.TextGrid',
 'D2002-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D2002-Pro.TextGrid',
 'M0009-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-M0009-Pro.TextGrid',
 'D0017-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D0017-Pro.TextGrid',
 'D2006-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D2006-Pro.TextGrid',
 'M0002-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-M0002-Pro.TextGrid',
 'D2009-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D2009-Pro.TextGrid',
 'M0023-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-M0023-Pro.TextGrid',
 'D0001-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D0001-Pro.TextGrid',
 'D0005-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D0005-Pro.TextGrid',
 'M0016-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-M0016-Pro.TextGrid',
 'M0003-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-M0003-Pro.TextGrid',
 'D0

w2vec +MFA

In [21]:
from collections import defaultdict
import re
import pandas as pd
style = pd.read_csv("/vol/corpora/Rhapsodie/wav_style.csv")
d={"file":[],"path":[]}
for hyp_path in full_paths:
    d["file"].append(hyp_path.stem.split("-")[0])
    d["path"].append(hyp_path)
df = pd.merge(pd.DataFrame(d), style, on="file")


In [29]:
ref_dict

{'D0020-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D0020-Pro.TextGrid',
 'D2002-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D2002-Pro.TextGrid',
 'M0009-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-M0009-Pro.TextGrid',
 'D0017-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D0017-Pro.TextGrid',
 'D2006-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D2006-Pro.TextGrid',
 'M0002-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-M0002-Pro.TextGrid',
 'D2009-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D2009-Pro.TextGrid',
 'M0023-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-M0023-Pro.TextGrid',
 'D0001-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D0001-Pro.TextGrid',
 'D0005-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-D0005-Pro.TextGrid',
 'M0016-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-M0016-Pro.TextGrid',
 'M0003-Rhap': '/vol/corpora/Rhapsodie/TextGrids-fev2013/Rhap-M0003-Pro.TextGrid',
 'D0

In [33]:

def get_ref_intervals_rhap(clean_ref, audio_path):
    """
    Corrects ref timestamps only when TextGrid has a genuine session-level offset
    (i.e. ref extends beyond audio duration by more than threshold seconds).
    No reference to hyp is used — fully independent correction.
    """
    audio, sr = sf.read(audio_path)
    audio_duration = len(audio) / sr
    ref_last = clean_ref[-1]["end"]
    ref_offset = clean_ref[0]["start"]
    
    return [{
            "phoneme": item["phoneme"],
            "start": item["start"] - ref_offset,
            "end": item["end"] - ref_offset
        } for item in clean_ref]

In [42]:

ref_inventory=set()
pred_inventory=set()
style_stats = defaultdict(lambda: {
    "S": 0,
    "D": 0,
    "I": 0,
    "N": 0,
    "boundary_errors": []
})
total_S = total_D = total_I = total_N = 0

all_start_errors = []
all_end_errors = []
all_duration_errors = []

mfa_phones=[]
# ------------------------------------------------------------------
# Main loop
# ------------------------------------------------------------------
for _,row in df.iterrows():
    hyp_path = row["path"]
    #stem = hyp_path.stem.split("-")[1]+"-"+hyp_path.stem.split("-")[0]
    stem=hyp_path.stem
    if stem not in ref_dict.keys():
        continue

    ref_path = ref_dict[stem]

    # --- Extract phones and timestamps ---
    phones_hyp, hyp_intervals = extract_phones_from_textgrid(
        hyp_path, t="phones"
    )

    phones_ref, ref_intervals = extract_phones_from_textgrid(
        ref_path, t="phone"
    )
    pred_alignments=[]
    for i,j,k in hyp_intervals:
        pred_alignments.append({"phoneme":k,"start":i,"end":j})
    ref_alignments=[]
    for i,j,k in ref_intervals:
        ref_alignments.append({"phoneme":k,"start":i,"end":j})
    
    # --- Normalize inventories ---
    ref_offset = ref_alignments[0]["start"]
    #hyp_offset = pred_alignments[0]["start"]

    ref_intervals = [{
            "phoneme": item["phoneme"],
            "start": item["start"] - ref_offset,
            "end": item["end"] - ref_offset} for item in ref_alignments]
    
    clean_ref,norm_ref = clean_alignment_dict(ref_intervals,flag="",is_hyp=False)
    ref_alignments=clean_ref
    
    phones_ref = norm_ref
    clean_hyp,norm_hyp = clean_alignment_dict(pred_alignments,"",is_hyp=True)
    pred_alignments=clean_hyp
    phones_hyp = norm_hyp
    pred_inventory.update(phones_hyp)
    ref_inventory.update(phones_ref)

    
    # --- PER computation ---
    ref_str = " ".join(phones_ref)
    hyp_str = " ".join(phones_hyp)
    mfa_phones.append(hyp_str)

    out = process_words(ref_str, hyp_str)

    style = row["style"]  # column already in your dataframe

    style_stats[style]["S"] += out.substitutions
    style_stats[style]["D"] += out.deletions
    style_stats[style]["I"] += out.insertions
    style_stats[style]["N"] += len(phones_ref)

    # --- Sequence alignment ---
    alignment = align_sequences(phones_ref, phones_hyp)

    # --- Boundary errors ---
    s_err, e_err, d_err = boundary_errors(
        ref_alignments,
        pred_alignments,
        alignment,phones_ref, phones_hyp
    )

    style_stats[style]["boundary_errors"].extend(s_err)
results = []

for style, stats in style_stats.items():
    S = stats["S"]
    D = stats["D"]
    I = stats["I"]
    N = stats["N"]

    per = (S + D + I) / N if N > 0 else 0

    errors = np.array(stats["boundary_errors"])

    if len(errors) > 0:
        mean_err = np.mean(errors)
        median_err = np.median(errors)
        max_err = np.max(errors)

        within_20ms = np.mean(errors <= 0.02) * 100
        within_50ms = np.mean(errors <= 0.05) * 100
    else:
        mean_err = median_err = max_err = 0
        within_20ms = within_50ms = 0

    results.append({
        "style": style,
        "PER": per*100,
        "Substitutions": S,
        "Deletions": D,
        "Insertions": I,
        "N_ref": N,
        "Mean_boundary_error": mean_err*1000,
        "Median_boundary_error": median_err*1000,
        "Max_boundary_error": max_err*1000,
        "%_within_20ms": within_20ms,
        "%_within_50ms": within_50ms
    })

results_df = pd.DataFrame(results)
results_df

{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}
{'ɥ'}


,style,PER,Substitutions,Deletions,Insertions,N_ref,Mean_boundary_error,Median_boundary_error,Max_boundary_error,%_within_20ms,%_within_50ms
0,spont,18.714273,3166,4679,798,46184,457.445956,54.6630,9893.739,37.437075,48.824956
1,planned,11.158240,1367,1527,368,29234,704.827793,227.6455,9104.298,23.997722,30.690964
2,semi,18.554807,848,1520,359,14697,91.735689,24.0010,5714.599,46.516344,67.710277


In [73]:
#results_df.to_csv("results_rhap_phone_gold.csv")

In [31]:
"""import re
# ------------------------------------------------------------------
# Containers
# ------------------------------------------------------------------

total_S = total_D = total_I = total_N = 0
d={"audio_filename":[],"ref_phonemes":[]}
all_start_errors = []
all_end_errors = []
all_duration_errors = []
ref_inventory=set()
pred_inventory=set()
mfa_phones=[]
# ------------------------------------------------------------------
# Main loop
# ------------------------------------------------------------------
for hyp_path in full_paths:
    #print(hyp_path)
    #rhap
    #stem = hyp_path.stem.split("-")[1]+"-"+hyp_path.stem.split("-")[0]
    stem=hyp_path.stem
    if stem not in ref_dict.keys():
        continue
    d["audio_filename"].append(stem)
    ref_path = ref_dict[stem]

    # --- Extract phones and timestamps ---
    phones_hyp, hyp_intervals = extract_phones_from_textgrid(
        hyp_path, t="phones"
    )
    #phones_ref, ref_intervals = extract_phones_from_textgrid(
        #ref_path,t="MAU"
    #)
    phones_ref, ref_intervals = extract_phones_from_textgrid_typaloc(
        ref_path
    )

    ref_alignments=[]
    for i,j,k in ref_intervals:
        ref_alignments.append({"phoneme":k,"start":i,"end":j})
    pred_alignments=[]
    for i,j,k in hyp_intervals:
        pred_alignments.append({"phoneme":k,"start":i,"end":j})
    
    clean_ref,norm_ref = clean_alignment_dict(ref_alignments,flag="typaloc",is_hyp=False)
    #clean_ref,norm_ref = clean_alignment_dict(ref_alignments,flag="",is_hyp=False)
    ref_alignments=clean_ref
    
    phones_ref = norm_ref
    clean_hyp,norm_hyp = clean_alignment_dict(pred_alignments,"",is_hyp=True)
    pred_alignments=clean_hyp
    phones_hyp = norm_hyp
    pred_inventory.update(phones_hyp)
    ref_inventory.update(phones_ref)
    d["ref_phonemes"].append(phones_ref)



    # --- Normalize inventories ---
    ref_offset = ref_alignments[0]["start"]
    hyp_offset = pred_alignments[0]["start"]

    ref_intervals = [{
            "phoneme": item["phoneme"],
            "start": item["start"] - ref_offset,
            "end": item["end"] - ref_offset} for item in ref_alignments]
    
    hyp_intervals = [{
            "phoneme": item["phoneme"],
            "start": item["start"] - hyp_offset,
            "end": item["end"] - hyp_offset} for item in pred_alignments]
    # --- PER computation ---
    ref_str = " ".join(phones_ref)
    hyp_str = " ".join(phones_hyp)

    out = process_words(ref_str, hyp_str)

    total_S += out.substitutions
    total_D += out.deletions
    total_I += out.insertions
    total_N += len(phones_ref)

    # --- Sequence alignment ---
    alignment = align_sequences(phones_ref, phones_hyp)

    # --- Boundary errors ---
    s_err, e_err, d_err = boundary_errors(
        ref_intervals,
        hyp_intervals,
        alignment,phones_ref, phones_hyp
    )

    all_start_errors.extend(s_err)
    all_end_errors.extend(e_err)
    all_duration_errors.extend(d_err)
    """

In [43]:
only_in_inv1 = pred_inventory - ref_inventory
only_in_inv2 = ref_inventory - pred_inventory
common = pred_inventory & ref_inventory

print("Only in inventory 1:", sorted(only_in_inv1))
print("Only in inventory 2:", sorted(only_in_inv2))
print("Common phones:", sorted(common))

Only in inventory 1: []
Only in inventory 2: ['ɥ']
Common phones: ['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'ŋ', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']


In [77]:
import pandas as pd
#pd.DataFrame(d).to_csv("/vol/experiments3/imbenamor/TAPAS-FRAIS/data/ctrl_phonemes_ref.csv")

In [78]:
# ------------------------------------------------------------------
# Final Metrics
# ------------------------------------------------------------------

corpus_per = 100 * (total_S + total_D + total_I) / total_N

mean_start = np.mean(all_start_errors) * 1000
median_start = np.median(all_start_errors) * 1000

def tolerance(errors, threshold_ms):
    return np.mean(
        [e <= threshold_ms/1000 for e in errors]
    ) * 100

print(f"\nCorpus PER: {corpus_per:.2f}%")
print(f"Mean boundary error: {mean_start:.2f} ms")
print(f"Median boundary error: {median_start:.2f} ms")
print(f"% within 20ms: {tolerance(all_start_errors, 20):.2f}%")
print(f"% within 50ms: {tolerance(all_start_errors, 50):.2f}%")



Corpus PER: 0.00%
Mean boundary error: 534.52 ms
Median boundary error: 20.00 ms
% within 20ms: 51.02%
% within 50ms: 68.58%
